# 02 — Exploratory data analysis

Each source and sheet is analyzed independently. Plots describe observed variables only; candidate SER values calculated as sphere + half-cylinder are explicitly exploratory because the supplied files do not document the clinical refraction convention. For `DataSet_main.csv`, `SE` remains uninterpreted pending its codebook.

In [1]:
from pathlib import Path
import shutil, subprocess, tempfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
ROOT=Path.cwd()
if not (ROOT/"DATA.xls").exists(): ROOT=ROOT.parent
OUT=ROOT/"outputs"/"eda"; OUT.mkdir(parents=True,exist_ok=True)
# Legacy workbook conversion is temporary and preserves DATA.xls.
import importlib.util
try:
    xls=pd.read_excel(ROOT/"DATA.xls",sheet_name=None)
except ImportError:
    exe=shutil.which("soffice") or "/Users/omvdangi/.cache/codex-runtimes/codex-primary-runtime/dependencies/bin/override/soffice"
    tmp=Path(tempfile.mkdtemp(prefix="myopia_eda_")); subprocess.run([exe,"--headless","--convert-to","xlsx","--outdir",str(tmp),str(ROOT/"DATA.xls")],check=True,capture_output=True)
    xls=pd.read_excel(tmp/"DATA.xlsx",sheet_name=None)
data3=pd.read_excel(ROOT/"Data3.xlsx",sheet_name=None)
csv=pd.read_csv(ROOT/"DataSet_main.csv",sep=";",decimal=",",engine="python")
datasets={"DATA.xls/2018":xls["2018"],"DATA.xls/2014":xls["2014"],"Data3.xlsx/2014-2017":data3["2014-2017"],"DataSet_main.csv":csv}
sns.set_theme(style="whitegrid",context="notebook")


In [2]:
# Cohort-level numeric summaries and candidate target summaries
summary=[]
for name,df in datasets.items():
    numeric=df.select_dtypes(include="number")
    numeric.describe().T.to_csv(OUT/(name.replace("/","_").replace(".","_")+"_descriptive.csv"))
    row={"dataset":name,"n":len(df),"p":df.shape[1]}
    sphere=[c for c in df if "spherical refraction of right eye" in str(c).lower()]
    cylinder=[c for c in df if "cyl" in str(c).lower() and "right" in str(c).lower()]
    if sphere and cylinder:
        s=pd.to_numeric(df[sphere[0]].astype(str).str.replace(",",".",regex=False),errors="coerce")
        c=pd.to_numeric(df[cylinder[0]].astype(str).str.replace(",",".",regex=False),errors="coerce")
        candidate=s+c/2; row.update(candidate_SER_n=int(candidate.notna().sum()),candidate_SER_mean=float(candidate.mean()),candidate_SER_sd=float(candidate.std()),candidate_SER_min=float(candidate.min()),candidate_SER_max=float(candidate.max()))
    if "SE" in df:
        se=pd.to_numeric(df["SE"].astype(str).str.replace(",",".",regex=False),errors="coerce")
        row.update(SE_candidate_n=int(se.notna().sum()),SE_candidate_mean=float(se.mean()),SE_candidate_sd=float(se.std()),SE_candidate_min=float(se.min()),SE_candidate_max=float(se.max()))
    summary.append(row)
summary=pd.DataFrame(summary); display(summary); summary.to_csv(OUT/"cohort_target_summary.csv",index=False)

,dataset,n,p,candidate_SER_n,candidate_SER_mean,candidate_SER_sd,candidate_SER_min,candidate_SER_max,SE_candidate_n,SE_candidate_mean,SE_candidate_sd,SE_candidate_min,SE_candidate_max
0,DATA.xls/2018,3658,11,3658.0,-1.620113,2.306414,-15.75,7.125,NaN,NaN,NaN,NaN,NaN
1,DATA.xls/2014,3057,11,3053.0,-1.817034,2.275324,-15.00,6.190,NaN,NaN,NaN,NaN,NaN
2,Data3.xlsx/2014-2017,2024,13,2011.0,-0.475107,1.219169,-8.00,5.625,NaN,NaN,NaN,NaN,NaN
3,DataSet_main.csv,876,23,NaN,NaN,NaN,NaN,NaN,874.0,-0.046373,1.85085,-9.54,8.46


In [3]:
# Histograms and boxplots for clinically named numeric fields, split by source.
field_patterns={"sphere":"spherical refraction","cylinder":"cylind","axial length":"axial length|^al of","height":"height","weight":"weight","age":"age"}
for name,df in datasets.items():
    for label,pat in field_patterns.items():
        cols=[c for c in df.columns if __import__("re").search(pat,str(c),flags=__import__("re").I)]
        for col in cols:
            vals=pd.to_numeric(df[col].astype(str).str.replace(",",".",regex=False),errors="coerce").dropna()
            if vals.empty: continue
            fig,ax=plt.subplots(1,2,figsize=(10,3.5))
            sns.histplot(vals,kde=True,ax=ax[0]); ax[0].set(title=f"{name}: {col}",xlabel=f"{col} (source units undocumented)",ylabel="Observations")
            sns.boxplot(x=vals,ax=ax[1]); ax[1].set(title="Distribution and potential extremes",xlabel=f"{col} (source units undocumented)")
            fig.tight_layout(); fig.savefig(OUT/(name.replace("/","_").replace(".","_")+f"_{label}.png"),dpi=220,bbox_inches="tight"); plt.close(fig)


In [4]:
# Correlations among numeric fields. Correlation is descriptive, not causal.
for name,df in datasets.items():
    num=df.select_dtypes(include="number").copy()
    if num.shape[1]<2: continue
    corr=num.corr(numeric_only=True)
    fig,ax=plt.subplots(figsize=(max(7,.8*len(corr.columns)),max(5,.65*len(corr.columns))))
    sns.heatmap(corr,cmap="vlag",center=0,annot=len(corr.columns)<=12,fmt=".2f",ax=ax)
    ax.set_title(f"{name}: pairwise numeric correlations")
    fig.tight_layout(); fig.savefig(OUT/(name.replace("/","_").replace(".","_")+"_correlation.png"),dpi=220,bbox_inches="tight"); plt.close(fig)


In [5]:
# Missingness visualisations and explicitly provisional right-eye candidate SER distributions.
import re
for name,df in datasets.items():
    miss=df.isna().mean().sort_values(ascending=False)
    if miss.max()>0:
        fig,ax=plt.subplots(figsize=(max(7,len(miss)*.45),3.5)); miss[miss>0].plot.bar(ax=ax,color="#4472C4")
        ax.set(title=f"{name}: fraction missing by field",ylabel="Fraction missing",xlabel="Source field",ylim=(0,1)); fig.tight_layout()
        fig.savefig(OUT/(name.replace("/","_").replace(".","_")+"_missingness.png"),dpi=220,bbox_inches="tight"); plt.close(fig)
    sphere=[c for c in df if "spherical refraction of right eye" in str(c).lower()]
    cylinder=[c for c in df if "cyl" in str(c).lower() and "right" in str(c).lower()]
    if sphere and cylinder:
        s=pd.to_numeric(df[sphere[0]].astype(str).str.replace(",",".",regex=False),errors="coerce")
        cy=pd.to_numeric(df[cylinder[0]].astype(str).str.replace(",",".",regex=False),errors="coerce")
        candidate=s+cy/2
        fig,ax=plt.subplots(figsize=(7,4)); sns.histplot(candidate.dropna(),kde=True,ax=ax,color="#4472C4")
        ax.set(title=f"{name}: provisional right-eye SER candidate (formula unverified)",xlabel="Sphere + cylinder/2 (diopters; provisional)",ylabel="Observations")
        fig.tight_layout(); fig.savefig(OUT/(name.replace("/","_").replace(".","_")+"_candidate_SER.png"),dpi=220,bbox_inches="tight"); plt.close(fig)
        # Candidate relationship with axial length only; no inference about causality.
        axcol=next((c for c in df if "axial length of right eye" in str(c).lower() or str(c).lower()=="al of right eye"),None)
        if axcol:
            x=pd.to_numeric(df[axcol].astype(str).str.replace(",",".",regex=False),errors="coerce")
            pair=pd.DataFrame({"candidate_SER":candidate,"axial_length":x}).dropna()
            if len(pair):
                fig,ax=plt.subplots(figsize=(6,4)); sns.regplot(data=pair,x="axial_length",y="candidate_SER",lowess=True,scatter_kws={"alpha":.25,"s":14},line_kws={"color":"darkred"},ax=ax)
                ax.set(title=f"{name}: exploratory candidate SER vs axial length",xlabel="Axial length (source units not documented)",ylabel="Candidate SER (D; convention unverified)")
                fig.tight_layout(); fig.savefig(OUT/(name.replace("/","_").replace(".","_")+"_candidate_SER_vs_AL.png"),dpi=220,bbox_inches="tight"); plt.close(fig)


In [6]:
# Code-frequency plots and exploratory feature-versus-provisional-target relationships.
for name,df in datasets.items():
    for col in df.columns:
        if df[col].nunique(dropna=True)<=20 and df[col].nunique(dropna=True)>1:
            counts=df[col].astype("string").fillna("<missing>").value_counts(dropna=False)
            fig,ax=plt.subplots(figsize=(max(5,.5*len(counts)),3.5)); counts.plot.bar(ax=ax,color="#70AD47")
            ax.set(title=f"{name}: observed category/code counts for {col}",xlabel=f"{col} (coding undocumented)",ylabel="Observations"); fig.tight_layout()
            fig.savefig(OUT/(name.replace("/","_").replace(".","_")+"_codes_"+str(col).replace(" ","_")+".png"),dpi=180,bbox_inches="tight"); plt.close(fig)
    sphere=next((c for c in df.columns if "spherical refraction of right eye" in str(c).lower()),None)
    cylinder=next((c for c in df.columns if "cyl" in str(c).lower() and "right" in str(c).lower()),None)
    if sphere and cylinder:
        y=pd.to_numeric(df[sphere].astype(str).str.replace(",",".",regex=False),errors="coerce")+pd.to_numeric(df[cylinder].astype(str).str.replace(",",".",regex=False),errors="coerce")/2
        for col in df.columns:
            if col in [sphere,cylinder] or pd.api.types.is_numeric_dtype(df[col]):
                x=pd.to_numeric(df[col],errors="coerce")
                if x.nunique(dropna=True)>3:
                    pair=pd.DataFrame({"predictor":x,"provisional_SER":y}).dropna()
                    if len(pair)>0:
                        fig,ax=plt.subplots(figsize=(5.5,3.5)); sns.regplot(data=pair,x="predictor",y="provisional_SER",lowess=True,scatter_kws={"alpha":.2,"s":10},ax=ax)
                        ax.set(title=f"{name}: provisional outcome vs {col}",xlabel=f"{col} (units/coding need verification)",ylabel="Candidate sphere + cylinder/2 (D; unverified)")
                        fig.tight_layout(); fig.savefig(OUT/(name.replace("/","_").replace(".","_")+"_candidate_vs_"+str(col).replace(" ","_")+".png"),dpi=180,bbox_inches="tight"); plt.close(fig)


## Interpretation and limitations
The workbook sheets represent different study periods, but participant overlap is unknown. No across-period inference should be made without identifiers and protocol details. There are no documented units for lifestyle time fields or coded categories. Outlying values are flagged visually for source review and are not removed. No missingness or anomaly pattern is automatically treated as invalid. Candidate SER plots are exploratory and do not establish the target clinically.